# Covariate screening v2 — adds occupancy and share_bus, with selection guardrails

Extends the univariate screening of the per-pair Weibull to two further covariates:
**occupancy** and **share_bus** (share of PCU from buses). Covariates are detected
dynamically, so the notebook screens whatever candidate columns exist in `data2.xlsx`
and reports which were found / missing. `share_bus` scale (fraction vs percent) is
auto-detected.

**Each covariate** is added singly to the pair's Weibull via the AFT scale
\(\lambda_i=\exp(\beta_0+\beta_1 z_i)\) and tested against the covariate-free Weibull
(likelihood-ratio test, AIC). Continuous effects are reported in natural units **and**
per +1 SD (so effects are comparable across covariates on different scales).

**Selection guardrails (stated a priori):**
* `improves_strict` = Yes only if **LR p < 0.05 AND ΔAIC ≥ 2**.
* per-pair covariate budget = ⌊N/15⌋ (limits how many covariates a small pair may take
  at the later integration stage).

A `Covariate_correlations` sheet reports Spearman correlations among the continuous
covariates (overall and within pair) — this documents the flow / speed-difference
interlinkage rather than assuming it. Output goes to the `Tables` folder.

In [1]:
# --- Cell 1: Imports and paths ---
import os
import numpy as np
import pandas as pd
from scipy import stats, optimize

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data2.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

MIN_BIN_GROUP = 5     # min count in each binary group to attempt a fit
DAIC_MIN      = 2.0   # strict-improvement AIC threshold

In [2]:
# --- Cell 2: Load, drop 2W, resolve which covariates are present ---
df = pd.read_excel(DATA_PATH)
df = df[df["V_Leading_Class"] != "2W"].copy()
df = df.rename(columns={"Time_Headway": "hw"})

def resolve_share_bus(frame):
    if "share_bus" in frame.columns:
        return "share_bus"
    hits = [c for c in frame.columns if "bus" in c.lower()]
    return hits[0] if hits else None

BUS_COL = resolve_share_bus(df)

# registry: name -> dict(col, type, unit, label)  (unit/label used for continuous)
REGISTRY = {
    "speed":      dict(col="Target_Speed_km/hr", type="cont", unit=1,    label="per +1 km/h"),
    "speed_diff": dict(col="Speed_Difference",   type="cont", unit=1,    label="per +1 km/h"),
    "flow":       dict(col="Flow_pcu/hr",        type="cont", unit=1000, label="per +1000 pcu/hr"),
    "off_cen":    dict(col="Off_centeredness",   type="bin",  unit=1,    label="True vs False"),
    "occupancy":  dict(col="Occupancy",          type="bin",  unit=1,    label="True vs False"),
}
if BUS_COL is not None:
    s = pd.to_numeric(df[BUS_COL], errors="coerce")
    if s.max() <= 1.5:      # looks like a 0-1 fraction
        REGISTRY["share_bus"] = dict(col=BUS_COL, type="cont", unit=0.1,  label="per +0.10 share")
    else:                   # looks like a percentage
        REGISTRY["share_bus"] = dict(col=BUS_COL, type="cont", unit=10.0, label="per +10 (pct pts)")

# keep only covariates whose column is actually present
CANDIDATES = {k: v for k, v in REGISTRY.items() if v["col"] in df.columns}
missing    = {k: v["col"] for k, v in REGISTRY.items() if v["col"] not in df.columns}

print("N =", len(df), "| pairs:", df["Pair"].nunique())
print("Screening covariates:", list(CANDIDATES.keys()))
if missing:
    print("Not found (skipped):", missing)
if BUS_COL:
    print(f"share_bus resolved to column '{BUS_COL}' ->", REGISTRY.get("share_bus", {}).get("label"))

N = 802 | pairs: 6
Screening covariates: ['speed', 'speed_diff', 'flow', 'off_cen', 'occupancy', 'share_bus']
share_bus resolved to column 'Share_Bus' -> per +0.10 share


In [3]:
# --- Cell 3: Weibull AFT likelihood + fitters ---
def _nll_cov(p, t, z):
    lp = stats.weibull_min.logpdf(t, np.exp(p[0]), loc=0, scale=np.exp(p[1] + p[2] * z))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def _nll_null(p, t):
    lp = stats.weibull_min.logpdf(t, np.exp(p[0]), loc=0, scale=np.exp(p[1]))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12

def fit_null(t):
    c0, _, s0 = stats.weibull_min.fit(t, floc=0)
    r = optimize.minimize(_nll_null, [np.log(c0), np.log(s0)], args=(t,),
                          method="Nelder-Mead", options={"maxiter": 8000})
    return r, (np.log(c0), np.log(s0))

def fit_cov(t, z, init):
    return optimize.minimize(_nll_cov, [init[0], init[1], 0.0], args=(t, z),
                             method="Nelder-Mead",
                             options={"maxiter": 8000, "xatol": 1e-7, "fatol": 1e-7})

In [4]:
# --- Cell 4: Screen every candidate against each pair's null Weibull ---
rows = []
for pair, g in df.groupby("Pair"):
    t = g["hw"].values
    n = len(t)
    r0, init = fit_null(t)
    ll0 = -r0.fun

    for name, meta in CANDIDATES.items():
        raw = pd.to_numeric(g[meta["col"]], errors="coerce").values.astype(float)

        if meta["type"] == "cont":
            sd = np.nanstd(raw)
            z = raw - np.nanmean(raw)
            enough = sd > 1e-9
        else:
            sd = np.nan
            z = raw
            enough = (len(np.unique(raw)) == 2 and
                      min((raw == 0).sum(), (raw == 1).sum()) >= MIN_BIN_GROUP)

        if not enough:
            rows.append(dict(Pair=pair, N=n, Covariate=name, Type=meta["type"],
                             Effect="-- low variation --", Effect_per_SD="",
                             coef_b1=np.nan, LR_chi2=np.nan, LR_p=np.nan,
                             dAIC=np.nan, improves="n/a", improves_strict="n/a"))
            continue

        r1 = fit_cov(t, z, init)
        ll1 = -r1.fun
        b1 = r1.x[2]
        LR = 2 * (ll1 - ll0)
        p_lr = stats.chi2.sf(LR, 1)
        dAIC = (2 * 2 - 2 * ll0) - (2 * 3 - 2 * ll1)

        if meta["type"] == "cont":
            eff  = f"{(np.exp(b1*meta['unit'])-1)*100:+.2f}%  {meta['label']}"
            effS = f"{(np.exp(b1*sd)-1)*100:+.2f}%  per +1 SD"
        else:
            eff  = f"{(np.exp(b1)-1)*100:+.2f}%  ({meta['label']})"
            effS = ""

        sig  = p_lr < 0.05
        rows.append(dict(Pair=pair, N=n, Covariate=name, Type=meta["type"],
                         Effect=eff, Effect_per_SD=effS, coef_b1=round(b1, 6),
                         LR_chi2=round(LR, 2),
                         LR_p="<0.001" if p_lr < 0.001 else round(p_lr, 4),
                         dAIC=round(dAIC, 2),
                         improves="Yes" if sig else "No",
                         improves_strict="Yes" if (sig and dAIC >= DAIC_MIN) else "No"))

screening = pd.DataFrame(rows)
screening

,Pair,N,Covariate,Type,Effect,Effect_per_SD,coef_b1,LR_chi2,LR_p,dAIC,improves,improves_strict
0,BTW_following_4W,250,speed,cont,-1.99% per +1 km/h,-10.37% per +1 SD,-0.020103,28.78,<0.001,26.78,Yes,Yes
1,BTW_following_4W,250,speed_diff,cont,-2.50% per +1 km/h,-11.44% per +1 SD,-0.025320,31.76,<0.001,29.76,Yes,Yes
2,BTW_following_4W,250,flow,cont,+2.71% per +1000 pcu/hr,+5.70% per +1 SD,0.000027,6.25,0.0124,4.25,Yes,Yes
3,BTW_following_4W,250,off_cen,bin,-9.21% (True vs False),,-0.096595,3.83,0.0504,1.83,No,No
4,BTW_following_4W,250,occupancy,bin,-7.03% (True vs False),,-0.072863,1.46,0.2275,-0.54,No,No
5,BTW_following_4W,250,share_bus,cont,-5.27% per +0.10 share,-1.65% per +1 SD,-0.541078,0.46,0.4964,-1.54,No,No
6,BTW_following_MT_3W,186,speed,cont,-3.15% per +1 km/h,-16.59% per +1 SD,-0.031974,36.26,<0.001,34.26,Yes,Yes
7,BTW_following_MT_3W,186,speed_diff,cont,-3.32% per +1 km/h,-15.63% per +1 SD,-0.033786,28.81,<0.001,26.81,Yes,Yes
8,BTW_following_MT_3W,186,flow,cont,+5.97% per +1000 pcu/hr,+13.49% per +1 SD,0.000058,16.13,<0.001,14.13,Yes,Yes
9,BTW_following_MT_3W,186,off_cen,bin,-4.92% (True vs False),,-0.050473,0.59,0.4438,-1.41,No,No


In [5]:
# --- Cell 5: dAIC and LR-p matrices ---
order_cov  = [c for c in CANDIDATES.keys()]
order_pair = (screening.drop_duplicates("Pair").sort_values("N", ascending=False)["Pair"].tolist())

num = screening.copy()
num["dAIC_num"] = pd.to_numeric(num["dAIC"], errors="coerce")
dAIC_matrix = (num.pivot(index="Pair", columns="Covariate", values="dAIC_num")
                  .reindex(index=order_pair, columns=order_cov).round(2))

def p_to_num(v):
    return 0.0005 if v == "<0.001" else (np.nan if pd.isna(v) else float(v))
num["p_num"] = num["LR_p"].map(p_to_num)
LRp_matrix = (num.pivot(index="Pair", columns="Covariate", values="p_num")
                 .reindex(index=order_pair, columns=order_cov))

print("dAIC (positive => covariate improves the per-pair Weibull):")
print(dAIC_matrix.to_string())
dAIC_matrix

dAIC (positive => covariate improves the per-pair Weibull):
Covariate             speed  speed_diff   flow  off_cen  occupancy  share_bus
Pair                                                                         
BTW_following_4W      26.78       29.76   4.25     1.83      -0.54      -1.54
BTW_following_MT_3W   34.26       26.81  14.13    -1.41       2.58      -1.24
BTW_following_NMT_3W   6.37       -0.29   3.77    -2.00      -2.00       0.12
PR_following_MT_3W    16.08        1.13   2.91    -1.93       9.22       1.80
PR_following_NMT_3W   -1.86       -1.22  -1.03    -1.38      -0.61      -1.02
PR_following_4W       -1.99       -1.99  -1.98     0.37      -1.79      -1.13


Covariate,speed,speed_diff,flow,off_cen,occupancy,share_bus
Pair,,,,,,
BTW_following_4W,26.78,29.76,4.25,1.83,-0.54,-1.54
BTW_following_MT_3W,34.26,26.81,14.13,-1.41,2.58,-1.24
BTW_following_NMT_3W,6.37,-0.29,3.77,-2.00,-2.00,0.12
PR_following_MT_3W,16.08,1.13,2.91,-1.93,9.22,1.80
PR_following_NMT_3W,-1.86,-1.22,-1.03,-1.38,-0.61,-1.02
PR_following_4W,-1.99,-1.99,-1.98,0.37,-1.79,-1.13


In [6]:
# --- Cell 6: Best covariate per pair (strict rule) + covariate budget ---
best = []
for pair in order_pair:
    sub = num[(num["Pair"] == pair) & (num["improves_strict"] == "Yes")]
    n = int(num[num["Pair"] == pair]["N"].iloc[0])
    budget = n // 15
    if sub.empty:
        best.append(dict(Pair=pair, N=n, cov_budget=budget,
                         Best_covariate="none (keep plain Weibull)",
                         dAIC=np.nan, LR_p=np.nan))
    else:
        top = sub.loc[sub["dAIC_num"].idxmax()]
        best.append(dict(Pair=pair, N=n, cov_budget=budget,
                         Best_covariate=top["Covariate"],
                         dAIC=round(top["dAIC_num"], 2), LR_p=top["LR_p"]))
best_per_pair = pd.DataFrame(best)
best_per_pair

,Pair,N,cov_budget,Best_covariate,dAIC,LR_p
0,BTW_following_4W,250,16,speed_diff,29.76,<0.001
1,BTW_following_MT_3W,186,12,speed,34.26,<0.001
2,BTW_following_NMT_3W,160,10,speed,6.37,0.0038
3,PR_following_MT_3W,104,6,speed,16.08,<0.001
4,PR_following_NMT_3W,59,3,none (keep plain Weibull),NaN,NaN
5,PR_following_4W,43,2,none (keep plain Weibull),NaN,NaN


In [7]:
# --- Cell 7: Correlation diagnostic among continuous covariates ---
cont_cols = {name: meta["col"] for name, meta in CANDIDATES.items() if meta["type"] == "cont"}

# overall Spearman matrix
C = df[list(cont_cols.values())].apply(pd.to_numeric, errors="coerce")
C.columns = list(cont_cols.keys())
corr_overall = C.corr(method="spearman").round(3)

# within-pair pairwise correlations (long form)
names = list(cont_cols.keys())
wp = []
for pair, g in df.groupby("Pair"):
    gg = g[list(cont_cols.values())].apply(pd.to_numeric, errors="coerce")
    gg.columns = names
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            r = stats.spearmanr(gg[names[i]], gg[names[j]], nan_policy="omit")[0]
            wp.append(dict(Pair=pair, var1=names[i], var2=names[j], spearman_r=round(r, 3)))
corr_within_pair = pd.DataFrame(wp)

print("Overall Spearman among continuous covariates:")
print(corr_overall.to_string())
corr_within_pair

Overall Spearman among continuous covariates:
            speed  speed_diff   flow  share_bus
speed       1.000       0.269 -0.368      0.054
speed_diff  0.269       1.000  0.118      0.019
flow       -0.368       0.118  1.000     -0.334
share_bus   0.054       0.019 -0.334      1.000


,Pair,var1,var2,spearman_r
0,BTW_following_4W,speed,speed_diff,0.089
1,BTW_following_4W,speed,flow,-0.410
2,BTW_following_4W,speed,share_bus,-0.008
3,BTW_following_4W,speed_diff,flow,0.071
4,BTW_following_4W,speed_diff,share_bus,-0.014
5,BTW_following_4W,flow,share_bus,-0.266
6,BTW_following_MT_3W,speed,speed_diff,0.296
7,BTW_following_MT_3W,speed,flow,-0.538
8,BTW_following_MT_3W,speed,share_bus,0.150
9,BTW_following_MT_3W,speed_diff,flow,0.074


In [8]:
# --- Cell 8: Save all outputs to the Tables folder ---
out_path = os.path.join(TABLES, "headway_covariate_screening_v2.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    screening.to_excel(xl,        sheet_name="Screening_long",   index=False)
    dAIC_matrix.to_excel(xl,      sheet_name="dAIC_matrix")
    LRp_matrix.round(4).to_excel(xl, sheet_name="LR_p_matrix")
    best_per_pair.to_excel(xl,    sheet_name="Best_per_pair",    index=False)
    corr_overall.to_excel(xl,     sheet_name="Corr_overall")
    corr_within_pair.to_excel(xl, sheet_name="Corr_within_pair", index=False)
print("Saved:", out_path)

Saved: D:\Headway\Tables\headway_covariate_screening_v2.xlsx
